In [2]:
#Baselines for each borough
spark.sql("""SELECT
    borough,
    AVG(complaint_count)    AS avg_complaints_per_hour,
    MAX(complaint_count)    AS max_complaints,
    MIN(complaint_count)    AS min_complaints,
    STDDEV(complaint_count)  AS std_dev,
    ROUND(MAX(complaint_count) * 1.0
        / NULLIF(AVG(complaint_count), 0), 1) AS max_to_avg_ratio
FROM gold_nyc_urban
GROUP BY borough
ORDER BY avg_complaints_per_hour DESC;""").show()

StatementMeta(, 5bb739d4-01c6-468a-90c3-25f523f6fd7e, 4, Finished, Available, Finished, False)

+-------------+-----------------------+--------------+--------------+-----------------+----------------+
|      borough|avg_complaints_per_hour|max_complaints|min_complaints|          std_dev|max_to_avg_ratio|
+-------------+-----------------------+--------------+--------------+-----------------+----------------+
|     Brooklyn|      133.7192192192192|           608|             0|73.90046176163446|             4.5|
|       Queens|     106.64001501501501|           458|             0|57.20751591995283|             4.3|
|        Bronx|      93.10923423423424|           294|             0|48.70985627878512|             3.2|
|    Manhattan|      83.24962462462463|           299|             0|45.03147395014573|             3.6|
|Staten Island|       20.7987987987988|           484|             0|33.60194277234447|            23.3|
+-------------+-----------------------+--------------+--------------+-----------------+----------------+



In [3]:
#Worst hour per borough?
spark.sql("""
SELECT
    borough,
    hour,
    day_of_week,
    complaint_count,
    weather_category,
    temperature_2m,
    top_complaint_type
FROM gold_nyc_urban g
WHERE complaint_count = (
    SELECT MAX(complaint_count)
    FROM gold_nyc_urban
    WHERE borough = g.borough
)
ORDER BY complaint_count DESC;""").show()

StatementMeta(, 5bb739d4-01c6-468a-90c3-25f523f6fd7e, 5, Finished, Available, Finished, False)

+-------------+-------------------+-----------+---------------+----------------+--------------+------------------+
|      borough|               hour|day_of_week|complaint_count|weather_category|temperature_2m|top_complaint_type|
+-------------+-------------------+-----------+---------------+----------------+--------------+------------------+
|     Brooklyn|2026-02-24 08:00:00|    Tuesday|            608|           Clear|          -4.2|       Snow or Ice|
|Staten Island|2026-02-24 08:00:00|    Tuesday|            484|           Clear|          -3.9|       Snow or Ice|
|       Queens|2026-02-24 08:00:00|    Tuesday|            458|           Clear|          -4.2|       Snow or Ice|
|    Manhattan|2026-02-24 09:00:00|    Tuesday|            299|           Clear|          -4.9|       Snow or Ice|
|        Bronx|2026-01-27 09:00:00|    Tuesday|            294|           Clear|         -12.2|    HEAT/HOT WATER|
+-------------+-------------------+-----------+---------------+----------------+

In [5]:
# Does traffic congestion affect some agencies' resolution time?
spark.sql("""
SELECT 
    h.borough,
    h.agency,
    h.agency_name,
    h.avg_hours_to_close as high_traffic_hours,
    l.avg_hours_to_close as low_traffic_hours,
    h.avg_hours_to_close - l.avg_hours_to_close as hours_difference,
    ROUND((h.avg_hours_to_close - l.avg_hours_to_close) * 100.0 
        / NULLIF(l.avg_hours_to_close, 0), 1) as pct_slower_in_high_traffic
FROM (
    SELECT g.borough, 'High Traffic' as traffic_level, s.agency, s.agency_name,
           COUNT(*) as total_complaints,
           AVG((unix_timestamp(s.closed_date) - unix_timestamp(s.created_date)) / 3600.0) as avg_hours_to_close
    FROM silver_stg_nyc_311 s
    JOIN gold_nyc_urban g
        ON s.borough = g.borough
        AND date_trunc('hour', s.created_date) = date_trunc('hour', g.hour)
    JOIN (SELECT borough, AVG(traffic_volume) as avg_traffic FROM gold_nyc_urban 
          WHERE traffic_volume IS NOT NULL AND traffic_volume > 0 GROUP BY borough) b 
        ON g.borough = b.borough
    WHERE g.traffic_volume IS NOT NULL AND g.traffic_volume > 0
    AND g.traffic_volume > b.avg_traffic
    AND s.closed_date IS NOT NULL AND s.created_date IS NOT NULL
    AND s.closed_date > s.created_date
    AND s.agency IN ('DSNY', 'HPD', 'NYPD')
    GROUP BY g.borough, s.agency, s.agency_name
) h
JOIN (
    SELECT g.borough, 'Low Traffic' as traffic_level, s.agency, s.agency_name,
           COUNT(*) as total_complaints,
           AVG((unix_timestamp(s.closed_date) - unix_timestamp(s.created_date)) / 3600.0) as avg_hours_to_close
    FROM silver_stg_nyc_311 s
    JOIN gold_nyc_urban g
        ON s.borough = g.borough
        AND date_trunc('hour', s.created_date) = date_trunc('hour', g.hour)
    JOIN (SELECT borough, AVG(traffic_volume) as avg_traffic FROM gold_nyc_urban 
          WHERE traffic_volume IS NOT NULL AND traffic_volume > 0 GROUP BY borough) b 
        ON g.borough = b.borough
    WHERE g.traffic_volume IS NOT NULL AND g.traffic_volume > 0
    AND g.traffic_volume <= b.avg_traffic
    AND s.closed_date IS NOT NULL AND s.created_date IS NOT NULL
    AND s.closed_date > s.created_date
    AND s.agency IN ('DSNY', 'HPD', 'NYPD')
    GROUP BY g.borough, s.agency, s.agency_name
) l ON h.borough = l.borough AND h.agency = l.agency
ORDER BY pct_slower_in_high_traffic DESC;
""").show()

StatementMeta(, 5bb739d4-01c6-468a-90c3-25f523f6fd7e, 7, Finished, Available, Finished, False)

+---------+------+--------------------+------------------+-----------------+----------------+--------------------------+
|  borough|agency|         agency_name|high_traffic_hours|low_traffic_hours|hours_difference|pct_slower_in_high_traffic|
+---------+------+--------------------+------------------+-----------------+----------------+--------------------------+
| Brooklyn|   HPD|Department of Hou...|    315.3991614281|   177.5234499558|  137.8757114723|                      77.7|
|Manhattan|   HPD|Department of Hou...|    267.4158054234|   154.0167867613|  113.3990186621|                      73.6|
|    Bronx|   HPD|Department of Hou...|    224.4079880344|   135.4859005820|   88.9220874524|                      65.6|
|   Queens|   HPD|Department of Hou...|    217.4569410754|   138.0963702239|   79.3605708515|                      57.5|
| Brooklyn|  NYPD|New York City Pol...|      2.7208529150|     2.5661396801|    0.1547132349|                       6.0|
| Brooklyn|  DSNY|Department of 

In [6]:
# Which Day of the Week Generates the Most Complaints Per Borough?
spark.sql("""
SELECT
    borough,
    day_of_week,
    SUM(complaint_count)                    AS total_complaints,
    RANK() OVER (
        PARTITION BY borough
        ORDER BY SUM(complaint_count) DESC
    )                                       AS day_rank
FROM gold_nyc_urban
WHERE complaints_available = 1
GROUP BY borough, day_of_week
ORDER BY borough, day_rank;""").show()

StatementMeta(, 5bb739d4-01c6-468a-90c3-25f523f6fd7e, 8, Finished, Available, Finished, False)

+---------+-----------+----------------+--------+
|  borough|day_of_week|total_complaints|day_rank|
+---------+-----------+----------------+--------+
|    Bronx|    Tuesday|           39729|       1|
|    Bronx|     Monday|           37462|       2|
|    Bronx|   Thursday|           35729|       3|
|    Bronx|  Wednesday|           35504|       4|
|    Bronx|     Friday|           35337|       5|
|    Bronx|     Sunday|           32405|       6|
|    Bronx|   Saturday|           31877|       7|
| Brooklyn|    Tuesday|           56189|       1|
| Brooklyn|     Monday|           53737|       2|
| Brooklyn|   Thursday|           52875|       3|
| Brooklyn|  Wednesday|           51248|       4|
| Brooklyn|     Friday|           51213|       5|
| Brooklyn|   Saturday|           46063|       6|
| Brooklyn|     Sunday|           44903|       7|
|Manhattan|     Friday|           33839|       1|
|Manhattan|    Tuesday|           33425|       2|
|Manhattan|   Thursday|           32775|       3|


In [7]:
spark.sql("""
--Most Frequent Named Complaint Type Per Borough; Where Should Each Agency Focus?
-- CTE: compute each SUM exactly once
WITH borough_totals AS (
    SELECT
        borough,
        SUM(noise_complaints)               AS noise,
        SUM(heat_hot_water_complaints)      AS heat_hotwater,
        SUM(street_complaints)              AS street,
        SUM(complaint_count)                AS total_complaints
    FROM gold_nyc_urban
    WHERE complaints_available = 1
    GROUP BY borough
)
-- Outer query: reference pre-computed values — no re-scanning
SELECT
    borough,
    noise,
    heat_hotwater,
    street,
    total_complaints,
    ROUND(100.0 * noise        / NULLIF(total_complaints, 0), 1)  AS noise_pct,
    ROUND(100.0 * heat_hotwater / NULLIF(total_complaints, 0), 1) AS heat_pct,
    ROUND(100.0 * street       / NULLIF(total_complaints, 0), 1)  AS street_pct,
    CASE
        WHEN noise >= heat_hotwater
         AND noise >= street        THEN 'Noise'
        WHEN heat_hotwater >= noise
         AND heat_hotwater >= street THEN 'Heat / Hot Water'
        ELSE                              'Street Condition'
    END                                                           AS top_named_complaint
FROM borough_totals
ORDER BY borough;""").show()

StatementMeta(, 5bb739d4-01c6-468a-90c3-25f523f6fd7e, 9, Finished, Available, Finished, False)

+-------------+-----+-------------+------+----------------+---------+--------+----------+-------------------+
|      borough|noise|heat_hotwater|street|total_complaints|noise_pct|heat_pct|street_pct|top_named_complaint|
+-------------+-----+-------------+------+----------------+---------+--------+----------+-------------------+
|        Bronx|44221|        61897| 45532|          248043|     17.8|    25.0|      18.4|   Heat / Hot Water|
|     Brooklyn|50323|        49569|111599|          356228|     14.1|    13.9|      31.3|   Street Condition|
|    Manhattan|45628|        42535| 31134|          221777|     20.6|    19.2|      14.0|              Noise|
|       Queens|36852|        23230|111531|          284089|     13.0|     8.2|      39.3|   Street Condition|
|Staten Island| 4054|         2125| 14168|           55408|      7.3|     3.8|      25.6|   Street Condition|
+-------------+-----+-------------+------+----------------+---------+--------+----------+-------------------+



In [9]:
#Complaint Closure Rate by Borough ; Which Borough Is Falling Behind?
spark.sql("""
SELECT
    borough,
    SUM(complaint_count)                                        AS total_complaints,
    SUM(num_closed_complaints)                                  AS total_closed,
    SUM(num_open_complaints)                                    AS total_open,
    ROUND(
        100.0 * SUM(num_closed_complaints)
        / NULLIF(SUM(complaint_count), 0)
    , 1)                                                        AS closure_rate_pct,
    ROUND(
        100.0 * SUM(num_open_complaints)
        / NULLIF(SUM(complaint_count), 0)
    , 1)                                                        AS open_rate_pct
FROM gold_nyc_urban
WHERE complaints_available = 1
GROUP BY borough
ORDER BY closure_rate_pct ASC;""").show()

StatementMeta(, 5bb739d4-01c6-468a-90c3-25f523f6fd7e, 11, Finished, Available, Finished, False)

+-------------+----------------+------------+----------+----------------+-------------+
|      borough|total_complaints|total_closed|total_open|closure_rate_pct|open_rate_pct|
+-------------+----------------+------------+----------+----------------+-------------+
|    Manhattan|          221777|      196637|     15762|            88.7|          7.1|
|        Bronx|          248043|      223270|     21048|            90.0|          8.5|
|     Brooklyn|          356228|      328026|     17919|            92.1|          5.0|
|       Queens|          284089|      262435|      7549|            92.4|          2.7|
|Staten Island|           55408|       52056|      1409|            94.0|          2.5|
+-------------+----------------+------------+----------+----------------+-------------+



In [10]:
# Peak Hour Staffing Alerts by Borough and Month ,Where and When Should the City Deploy Resources?
spark.sql("""
WITH ranked_complaints AS (
    SELECT
        borough,
        hour_of_day,
        complaint_count,
        PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY complaint_count)
            OVER (PARTITION BY borough, hour_of_day) AS p90_complaints
    FROM gold_nyc_urban
    WHERE complaints_available = 1
),
aggregated AS (
    SELECT DISTINCT
        borough,
        hour_of_day,
        p90_complaints
    FROM ranked_complaints
)
SELECT
    borough,
    hour_of_day,
    CAST(p90_complaints AS DECIMAL(6,1))    AS p90_complaints,
    CASE
        WHEN p90_complaints > 150 THEN 'CRITICAL — staff peak'
        WHEN p90_complaints > 100 THEN 'HIGH — monitor closely'
        WHEN p90_complaints > 50  THEN 'MODERATE'
        ELSE                           'LOW'
    END                                     AS staffing_alert
FROM aggregated
ORDER BY borough, p90_complaints DESC;""").show()

StatementMeta(, 5bb739d4-01c6-468a-90c3-25f523f6fd7e, 12, Finished, Available, Finished, False)

+-------+-----------+--------------+--------------------+
|borough|hour_of_day|p90_complaints|      staffing_alert|
+-------+-----------+--------------+--------------------+
|  Bronx|         12|         192.9|CRITICAL — staff ...|
|  Bronx|         10|         191.3|CRITICAL — staff ...|
|  Bronx|         11|         188.1|CRITICAL — staff ...|
|  Bronx|         13|         186.9|CRITICAL — staff ...|
|  Bronx|         16|         186.3|CRITICAL — staff ...|
|  Bronx|          9|         184.9|CRITICAL — staff ...|
|  Bronx|         14|         175.9|CRITICAL — staff ...|
|  Bronx|          8|         166.2|CRITICAL — staff ...|
|  Bronx|         17|         163.9|CRITICAL — staff ...|
|  Bronx|         15|         162.2|CRITICAL — staff ...|
|  Bronx|         20|         157.3|CRITICAL — staff ...|
|  Bronx|         18|         155.3|CRITICAL — staff ...|
|  Bronx|         19|         151.6|CRITICAL — staff ...|
|  Bronx|          7|         144.6|HIGH — monitor cl...|
|  Bronx|     